# Video Game Sales Analysis
## Notebook 5: Feature Engineering

### Purpose

This notebook transforms the findings from the advanced exploratory analysis into model-ready features for predicting global video game sales.

The feature engineering process focuses on preparing categorical and numerical variables, reducing sparse categories, evaluating interaction features, and preventing data leakage. All preprocessing decisions that learn information from the data will be based on the training set and then applied consistently to unseen data.

### Feature Engineering Objectives

- Define the predictor variables and target variable.
- Exclude regional sales variables to prevent target leakage.
- Split the data into training and test sets before learned preprocessing.
- Encode categorical features such as Genre and Platform.
- Reduce Publisher sparsity by grouping rare publishers.
- Evaluate a Genre–Platform interaction feature.
- Prepare Year appropriately for modeling.
- Evaluate the log-transformed Global Sales target.
- Build a reproducible preprocessing workflow for machine learning.

## 1. Import Libraries

The required libraries are imported for data manipulation, train-test splitting, categorical encoding, and construction of the feature preprocessing pipeline.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 2. Load the Cleaned Dataset

The cleaned video game sales dataset produced during the earlier data preparation stage is loaded for feature engineering.

In [2]:
df = pd.read_csv("vgsales_clean.csv")

df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


## 3. Define Predictor Variables and Target

The predictive modeling objective is to estimate a game's `Global_Sales` using information available about the game.

Regional sales variables (`NA_Sales`, `EU_Sales`, `JP_Sales`, and `Other_Sales`) are excluded because `Global_Sales` is derived from these values. Including them as predictors would introduce target leakage.

The initial predictor variables selected from the advanced exploratory analysis are:

- `Platform`
- `Year`
- `Genre`
- `Publisher`

The target variable is `Global_Sales`. Game `Name` is excluded because it contains a very large number of unique categories and is not suitable as a direct predictor in the initial model.

In [3]:
X = df[
    ["Platform", "Year", "Genre", "Publisher"]
].copy()

y = df["Global_Sales"].copy()

In [4]:
X.head()

,Platform,Year,Genre,Publisher
0,Wii,2006,Sports,Nintendo
1,NES,1985,Platform,Nintendo
2,Wii,2008,Racing,Nintendo
3,Wii,2009,Sports,Nintendo
4,GB,1996,Role-Playing,Nintendo


In [5]:
y.head()

0    82.74
1    40.24
2    35.82
3    33.00
4    31.37
Name: Global_Sales, dtype: float64

## 4. Train-Test Split

The dataset is divided into training and test sets before performing learned feature engineering. This prevents information from the test set from influencing preprocessing decisions and helps ensure that model evaluation reflects performance on genuinely unseen data.

An 80/20 split is used, with 80% of the observations used for training and 20% reserved for final testing.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [7]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (13032, 4)
X_test: (3259, 4)
y_train: (13032,)
y_test: (3259,)


## 5. Rare Publisher Identification

The exploratory analysis showed that `Publisher` is a high-cardinality categorical variable with many publishers represented by only a small number of games.

To reduce sparsity, publishers with limited representation will eventually be grouped into an `Other` category. To prevent data leakage, publisher frequencies are calculated using the training data only. The resulting rule will then be applied consistently to both the training and test sets.

In [8]:
publisher_counts_train = X_train["Publisher"].value_counts()

publisher_counts_train.head(10)

Publisher
Electronic Arts                 1073
Activision                       767
Namco Bandai Games               748
Ubisoft                          728
Konami Digital Entertainment     660
THQ                              570
Nintendo                         564
Sony Computer Entertainment      544
Sega                             513
Take-Two Interactive             337
Name: count, dtype: int64

In [9]:
rare_publishers = publisher_counts_train[
    publisher_counts_train < 10
].index

len(rare_publishers)

388

In [10]:
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

In [11]:
def group_rare_publishers(publisher):
    if publisher in rare_publishers:
        return "Other"
    else:
        return publisher

In [12]:
X_train_fe["Publisher"] = X_train_fe["Publisher"].apply(
    group_rare_publishers
)

In [15]:
print("Before:", X_train["Publisher"].nunique())
print("After:", X_train_fe["Publisher"].nunique())

Before: 528
After: 141


In [16]:
X_train_fe["Publisher"].value_counts().head(10)

Publisher
Electronic Arts                 1073
Other                            999
Activision                       767
Namco Bandai Games               748
Ubisoft                          728
Konami Digital Entertainment     660
THQ                              570
Nintendo                         564
Sony Computer Entertainment      544
Sega                             513
Name: count, dtype: int64

In [17]:
def group_test_publishers(publisher):
    if publisher in known_publishers:
        return publisher
    else:
        return "Other"

In [18]:
known_publishers = publisher_counts_train[
    publisher_counts_train >= 10
].index

In [19]:
X_test_fe["Publisher"] = X_test_fe["Publisher"].apply(
    group_test_publishers
)

In [20]:
print("Training publisher categories:",
      X_train_fe["Publisher"].nunique())

print("Test publisher categories:",
      X_test_fe["Publisher"].nunique())

print("Test observations grouped as Other:",
      (X_test_fe["Publisher"] == "Other").sum())

Training publisher categories: 141
Test publisher categories: 136
Test observations grouped as Other: 274


## 6. Create Genre–Platform Interaction Feature

The advanced exploratory analysis suggested that sales performance may depend on specific combinations of genre and platform. A combined `Genre_Platform` feature is therefore created to allow future models to evaluate whether these interactions provide additional predictive information.

In [21]:
X_train_fe["Genre_Platform"] = (
    X_train_fe["Genre"] + "_" + X_train_fe["Platform"]
)

X_test_fe["Genre_Platform"] = (
    X_test_fe["Genre"] + "_" + X_test_fe["Platform"]
)

In [22]:
X_train_fe[
    ["Genre", "Platform", "Genre_Platform"]
].head()

,Genre,Platform,Genre_Platform
1669,Fighting,PSP,Fighting_PSP
653,Sports,PS4,Sports_PS4
3703,Shooter,PS2,Shooter_PS2
12196,Racing,X360,Racing_X360
7224,Shooter,GBA,Shooter_GBA


## 7. Define Feature Types

The engineered predictor variables are separated into categorical and numerical features so that appropriate preprocessing can be applied to each type.

Categorical variables will be one-hot encoded, while `Year` will initially remain numerical.


In [23]:
categorical_features = [
    "Platform",
    "Genre",
    "Publisher",
    "Genre_Platform"
]

numerical_features = [
    "Year"
]

## 8. Build the Preprocessing Transformer

A `ColumnTransformer` is used to apply different preprocessing steps to different feature types. Categorical variables are one-hot encoded, while the numerical `Year` feature is passed through unchanged.

The transformer is fit using the training data only to prevent information from the test set from influencing preprocessing.

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numerical_features
        )
    ]
)

In [27]:
preprocessor.fit(X_train_fe)

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [28]:
X_train_processed = preprocessor.transform(X_train_fe)
X_test_processed = preprocessor.transform(X_test_fe)

In [29]:
feature_names = preprocessor.get_feature_names_out()

feature_names[:20]

array(['cat__Platform_2600', 'cat__Platform_3DO', 'cat__Platform_3DS',
       'cat__Platform_DC', 'cat__Platform_DS', 'cat__Platform_GB',
       'cat__Platform_GBA', 'cat__Platform_GC', 'cat__Platform_GEN',
       'cat__Platform_GG', 'cat__Platform_N64', 'cat__Platform_NES',
       'cat__Platform_NG', 'cat__Platform_PC', 'cat__Platform_PCFX',
       'cat__Platform_PS', 'cat__Platform_PS2', 'cat__Platform_PS3',
       'cat__Platform_PS4', 'cat__Platform_PSP'], dtype=object)

In [30]:
len(feature_names)

476

In [31]:
print("Original feature count:", X_train_fe.shape[1])
print("Processed feature count:", X_train_processed.shape[1])

Original feature count: 5
Processed feature count: 476


### Preprocessing Output

The feature engineering process expanded the original 5 predictor variables into 476 model-ready features. Most of this increase is caused by one-hot encoding the categorical variables, particularly `Publisher` and the engineered `Genre_Platform` interaction feature.

The earlier rare-publisher grouping helped reduce unnecessary sparse categories before encoding. The resulting feature matrix will be evaluated during modeling to determine whether the additional categorical and interaction features improve predictive performance.

## 9. Evaluate Interaction Feature Complexity

The `Genre_Platform` interaction may capture useful combined effects, but it also increases the dimensionality of the encoded feature space. To understand this tradeoff, preprocessing is compared with and without the interaction feature.

In [32]:
categorical_features_base = [
    "Platform",
    "Genre",
    "Publisher"
]

numerical_features = [
    "Year"
]

In [33]:
preprocessor_base = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features_base
        ),
        (
            "num",
            "passthrough",
            numerical_features
        )
    ]
)

In [34]:
preprocessor_base.fit(X_train_fe)

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [35]:
base_feature_names = preprocessor_base.get_feature_names_out()

print("Without interaction:", len(base_feature_names))
print("With interaction:", len(feature_names))
print("Extra features added:",
      len(feature_names) - len(base_feature_names))

Without interaction: 185
With interaction: 476
Extra features added: 291


### Interaction Feature Complexity Interpretation

Adding the `Genre_Platform` interaction increases the encoded feature space from 185 to 476 features, introducing 291 additional predictors.

Although the exploratory analysis suggested that certain Genre–Platform combinations may contain useful information about sales performance, the interaction substantially increases model complexity. Therefore, the interaction feature will not automatically be assumed to improve prediction.

During model development, performance will be compared between a base feature set without the interaction and an expanded feature set containing `Genre_Platform`. The interaction will be retained only if it provides meaningful improvement in predictive performance.

## 10. Prepare Alternative Target Variable

The advanced exploratory analysis showed that `Global_Sales` is strongly right-skewed because a relatively small number of blockbuster games achieve exceptionally high sales.

Rather than removing these legitimate observations, a log-transformed version of the target is created using `np.log1p()`. This transformation reduces the influence of extreme values while safely handling sales values equal to zero.

Both the original and log-transformed targets will be retained so their predictive performance can be compared during model development.

In [36]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [37]:
print("Original target:")
print(y_train.head())

print("\nLog-transformed target:")
print(y_train_log.head())

Original target:
1669     1.19
653      2.38
3703     0.53
12196    0.06
7224     0.21
Name: Global_Sales, dtype: float64

Log-transformed target:
1669     0.783902
653      1.217876
3703     0.425268
12196    0.058269
7224     0.190620
Name: Global_Sales, dtype: float64


## 11. Feature Engineering Summary

The feature engineering stage transformed the exploratory findings into model-ready predictors while taking steps to prevent data leakage and reduce unnecessary feature sparsity.

### Predictor Selection

The initial predictors selected for modeling are:

- `Platform`
- `Genre`
- `Publisher`
- `Year`

Regional sales variables (`NA_Sales`, `EU_Sales`, `JP_Sales`, and `Other_Sales`) were excluded because they are components of `Global_Sales` and would introduce target leakage.

### Train-Test Separation

The data was divided into training and test sets before learned preprocessing decisions were made. This ensures that the test set remains unseen and provides a more reliable evaluation of model generalization.

### Publisher Rare-Category Handling

Publisher frequencies were learned from the training data only. Publishers appearing fewer than 10 times in the training set were grouped into an `Other` category.

This reduced the number of publisher categories in the training data from 528 to 141 and reduced the number of sparse features created during one-hot encoding.

### Genre–Platform Interaction

A `Genre_Platform` interaction feature was created based on patterns identified during exploratory analysis. Including this feature increased the encoded feature space from 185 to 476 predictors.

Because this represents a substantial increase in model complexity, models with and without the interaction feature will be compared before deciding whether it should be retained.

### Categorical Encoding

Categorical predictors were prepared using one-hot encoding with `handle_unknown="ignore"`. The encoder was fit using the training data only so that unseen test categories do not influence the learned preprocessing structure.

### Target Transformation

Both the original `Global_Sales` target and a log-transformed version created with `np.log1p()` were retained. The log transformation reduces the influence of extreme blockbuster sales while preserving all observations.

Model performance using the original and transformed targets will be compared during the machine learning stage.

## Next Step: Machine Learning

The feature engineering process has produced two candidate feature configurations:

1. **Base Feature Set:** Platform, Genre, Publisher, and Year.
2. **Expanded Feature Set:** Base features plus the Genre–Platform interaction.

Both configurations can be evaluated using the original and log-transformed Global Sales targets.

The next stage will establish baseline regression performance and compare multiple machine learning algorithms using consistent evaluation procedures.